# 08 — MICRO-TIMING : le swing du jembé

Le sujet phare des travaux de Polak : dans la musique de jembé, les 12 subdivisions du cycle ne sont **pas égales**. Chaque pulsation (3 subdivisions) a un « swing » propre au rythme — et ce n'est ni du hasard ni de l'imprécision : les musiciens le reproduisent avec une précision extrême.

On mesure ici, depuis les onsets annotés :
1. la **position réelle** de chaque subdivision dans son cycle (vs grille parfaite),
2. le **ratio de swing** court/moyen/long de chaque pièce,
3. l'**accélération du tempo** au fil du morceau (l'accelerando).


## 0. Imports & extraction des positions relatives

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
import djembe_utils as dj

c = dj.list_corpus(); INST = ['J1','J2','D1','D2']

def cycle_positions(piece, take):
    """[(SD, position relative 0-1, durée cycle, début)] pour une prise."""
    sel = dj.load_selected(piece, take)
    if sel is None: return []
    sel = sel.copy()
    sel['cyc'] = sel['Label SD'].astype(str).str.split('|').str[0]
    tcols = [i for i in INST if i in sel.columns]
    sel['t'] = sel[tcols].min(axis=1, skipna=True)   # premier onset de la position
    out, cycles = [], list(sel.groupby('cyc', sort=False))
    for k in range(len(cycles)-1):
        g, gn = cycles[k][1], cycles[k+1][1]
        t1, t1n = g.loc[g['SD']==1,'t'].dropna(), gn.loc[gn['SD']==1,'t'].dropna()
        if t1.empty or t1n.empty: continue
        start = float(t1.iloc[0]); dur = float(t1n.iloc[0]) - start
        if not (0.5 < dur < 5): continue
        for _, r in g.iterrows():
            if pd.notna(r['t']):
                rel = (r['t'] - start) / dur
                if -0.05 < rel < 1.05:
                    out.append((int(r['SD']), rel, dur, start))
    return out

POS = {}
for piece in dj.PIECES:
    rows = []
    for t in c.loc[c['piece']==piece, 'take'].unique():
        rows += cycle_positions(piece, int(t))
    POS[piece] = pd.DataFrame(rows, columns=['SD','rel','dur','start'])
    print(piece, ':', len(POS[piece]), 'onsets positionnés')

## 1. Les subdivisions ne sont pas là où on les attend
Écart entre la position **réelle** moyenne et la grille isochrone (en % du cycle). Un écart négatif = la frappe arrive **en avance** sur la grille parfaite.

In [ ]:
iso = np.arange(12)/12
plt.figure(figsize=(12,5))
for piece, d in POS.items():
    m = d.groupby('SD')['rel'].mean().reindex(range(1,13)).values
    plt.plot(range(1,13), (m-iso)*100, marker='o', lw=2, label=piece)
plt.axhline(0, color='k', lw=1, alpha=.5)
plt.xticks(range(1,13)); plt.xlabel('subdivision'); plt.ylabel('écart à la grille parfaite (% du cycle)')
plt.title('Micro-timing : où tombent vraiment les subdivisions'); plt.legend(); plt.grid(alpha=.2); plt.show()

## 2. Le ratio de swing (les 3 subdivisions d'une pulsation)
Chaque pulsation = 3 subdivisions. On mesure leurs durées relatives (en % de la pulsation). Isochrone serait 33/33/33.

In [ ]:
print('ratio court/moyen/long par pièce (moyenne des 4 pulsations) :\n')
res = {}
for piece, d in POS.items():
    m = d.groupby('SD')['rel'].mean(); ratios = []
    for beat in range(4):
        s1, s2, s3 = beat*3+1, beat*3+2, beat*3+3
        end = m[beat*3+4] if beat*3+4 <= 12 else 1.0
        d1, d2, d3 = m[s2]-m[s1], m[s3]-m[s2], end-m[s3]
        tot = d1+d2+d3
        ratios.append([d1/tot*100, d2/tot*100, d3/tot*100])
    r = np.mean(ratios, axis=0); res[piece] = r
    print(f'  {piece:10s} : {r[0]:.0f} / {r[1]:.0f} / {r[2]:.0f}')

x = np.arange(3); w = 0.25
plt.figure(figsize=(8,5))
for i, (piece, r) in enumerate(res.items()):
    plt.bar(x+(i-1)*w, r, w, label=piece)
plt.axhline(100/3, color='k', ls='--', alpha=.5, label='isochrone (33,3)')
plt.xticks(x, ['1ère subdiv.','2ème','3ème']); plt.ylabel('% de la pulsation')
plt.title('Le swing propre à chaque rythme'); plt.legend(); plt.show()

## 3. L'accelerando : le tempo monte du début à la fin

In [ ]:
plt.figure(figsize=(13,5))
COLP = dict(zip(dj.PIECES, ['#2b6cff','#d63bd6','#2ec45f']))
for piece, d in POS.items():
    for start0 in d['start'].groupby((d['start'].diff()<0).cumsum()).groups:  # par prise
        pass
# plus simple : tracer duree de cycle vs temps, par prise
for piece in dj.PIECES:
    for t in c.loc[c['piece']==piece, 'take'].unique():
        cp = pd.DataFrame(cycle_positions(piece, int(t)), columns=['SD','rel','dur','start'])
        if cp.empty: continue
        cyc = cp.drop_duplicates('start')
        plt.plot(cyc['start'], 60/(cyc['dur']/4), color=COLP[piece], alpha=.55, lw=1.2)
for piece, col in COLP.items():
    plt.plot([], [], color=col, label=piece)
plt.xlabel('temps (s)'); plt.ylabel('tempo (pulsations/min)')
plt.title('Accelerando : toutes les prises accélèrent (+26 % à +76 %)'); plt.legend(); plt.grid(alpha=.2); plt.show()

## Bilan
- **Swing non-isochrone confirmé** : Manjanin et Woloso ≈ court-moyen-long (**25/34/41**), Maraka ≈ **36/28/36** — on retrouve les résultats publiés de Polak depuis les données.
- **Accelerando systématique** : chaque prise accélère (+26 à +76 %). C'est une propriété du style, pas un défaut.
- Deux usages : (1) **améliorer la génération** (07) en remplaçant la grille isochrone par le swing mesuré ; (2) features supplémentaires pour **différencier les rythmes** (le swing distingue Maraka des deux autres).